In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

cwd = Path.cwd()

if (cwd / "component4").exists():
    ROOT = cwd
elif cwd.name == "notebooks" and cwd.parent.name == "component4":
    ROOT = cwd.parent.parent
else:
    ROOT = cwd

sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "component4" / "data" / "mock"
OUTPUT_DIR = ROOT / "component4" / "outputs"

print("Project root:", ROOT)
print("Data directory exists:", DATA_DIR.exists())

Project root: e:\J26-DS-349
Data directory exists: True


In [2]:
from component4.src.preference_uncertainty import (
    sample_preferences
)

from component4.src.evidence_uncertainty import (
    weighted_bootstrap_evidence
)

print("C4 uncertainty modules imported successfully.")

C4 uncertainty modules imported successfully.


In [3]:
products = pd.read_csv(DATA_DIR / "products.csv")
preferences = pd.read_csv(DATA_DIR / "user_preferences.csv")
c1 = pd.read_csv(DATA_DIR / "c1_product_aspects.csv")
c2 = pd.read_csv(DATA_DIR / "c2_review_evidence.csv")
c3 = pd.read_csv(DATA_DIR / "c3_temporal_features.csv")

print("Products:", products.shape)
print("Preferences:", preferences.shape)
print("C1:", c1.shape)
print("C2:", c2.shape)
print("C3:", c3.shape)

Products: (5, 4)
Preferences: (6, 5)
C1: (30, 5)
C2: (90, 5)
C3: (30, 6)


In [4]:
features = c1.merge(
    c3,
    on=["product_id", "aspect"],
    how="inner"
)

features["nominal_aspect_score"] = (
    features["aspect_quality"]
    + features["recent_quality"]
) / 2

display(
    features[
        [
            "product_id",
            "aspect",
            "nominal_aspect_score"
        ]
    ].head(10)
)

,product_id,aspect,nominal_aspect_score
0,P001,battery,0.890
1,P001,camera,0.735
2,P001,performance,0.905
3,P001,display,0.815
4,P001,durability,0.785
5,P001,price,0.695
6,P002,battery,0.735
7,P002,camera,0.900
8,P002,performance,0.830
9,P002,display,0.855


In [5]:
nominal_preferences = preferences[
    ["aspect", "importance", "confidence"]
].copy()

nominal_preferences["nominal_weight"] = (
    nominal_preferences["importance"]
    / nominal_preferences["importance"].sum()
)

display(nominal_preferences)

print(
    "Weight sum:",
    nominal_preferences["nominal_weight"].sum()
)

,aspect,importance,confidence,nominal_weight
0,battery,5,0.90,0.238095
1,camera,3,0.70,0.142857
2,performance,4,0.85,0.190476
3,display,3,0.80,0.142857
4,durability,2,0.75,0.095238
5,price,4,0.90,0.190476


Weight sum: 0.9999999999999999


In [6]:
def calculate_nominal_evidence(c2_data):

    rows = []

    for (product_id, aspect), group in c2_data.groupby(
        ["product_id", "aspect"]
    ):

        quality = group["evidence_quality"].to_numpy(
            dtype=float
        )

        weights = np.clip(
            quality,
            0,
            None
        )

        if weights.sum() == 0:
            reliability = quality.mean()
        else:
            reliability = np.average(
                quality,
                weights=weights
            )

        rows.append({
            "product_id": product_id,
            "aspect": aspect,
            "nominal_evidence_reliability": reliability
        })

    return pd.DataFrame(rows)


nominal_evidence = calculate_nominal_evidence(c2)

display(nominal_evidence.head(10))

,product_id,aspect,nominal_evidence_reliability
0,P001,battery,0.849841
1,P001,camera,0.783793
2,P001,display,0.789660
3,P001,durability,0.792298
4,P001,performance,0.865564
5,P001,price,0.785726
6,P002,battery,0.692157
7,P002,camera,0.833158
8,P002,display,0.789660
9,P002,durability,0.792298


In [7]:
nominal = features.merge(
    nominal_preferences[
        ["aspect", "nominal_weight"]
    ],
    on="aspect",
    how="inner"
)

nominal["weighted_contribution"] = (
    nominal["nominal_aspect_score"]
    * nominal["nominal_weight"]
)

nominal_scores = (
    nominal
    .groupby("product_id", as_index=False)
    ["weighted_contribution"]
    .sum()
    .rename(
        columns={
            "weighted_contribution":
            "personalized_score"
        }
    )
)

nominal_scores = nominal_scores.merge(
    products,
    on="product_id",
    how="left"
)

max_budget = preferences["max_budget"].iloc[0]

nominal_scores = (
    nominal_scores[
        nominal_scores["price"] <= max_budget
    ]
    .sort_values(
        "personalized_score",
        ascending=False
    )
    .reset_index(drop=True)
)

nominal_scores["rank"] = np.arange(
    1,
    len(nominal_scores) + 1
)

display(
    nominal_scores[
        [
            "rank",
            "product_id",
            "product_name",
            "personalized_score"
        ]
    ]
)

nominal_winner = nominal_scores.iloc[0]["product_id"]

print("Nominal winner:", nominal_winner)

,rank,product_id,product_name,personalized_score
0,1,P005,Phone E,0.823095
1,2,P001,Phone A,0.812857
2,3,P002,Phone B,0.811429
3,4,P004,Phone D,0.777619


Nominal winner: P005


In [8]:
N_SIMULATIONS = 500

preference_samples = sample_preferences(
    preferences,
    n_samples=N_SIMULATIONS,
    uncertainty_scale=1.0,
    random_state=42
)

print(
    "Preference sample rows:",
    len(preference_samples)
)

Preference sample rows: 3000


In [9]:
evidence_samples = weighted_bootstrap_evidence(
    c2,
    n_bootstrap=N_SIMULATIONS,
    random_state=42
)

evidence_samples = evidence_samples.rename(
    columns={
        "bootstrap_id": "simulation_id"
    }
)

print(
    "Evidence sample rows:",
    len(evidence_samples)
)

Evidence sample rows: 15000


In [10]:
preference_mode = features.merge(
    preference_samples[
        [
            "simulation_id",
            "aspect",
            "sampled_weight"
        ]
    ],
    on="aspect",
    how="inner"
)

preference_mode["adjusted_aspect_score"] = (
    preference_mode["nominal_aspect_score"]
)

preference_mode["weighted_contribution"] = (
    preference_mode["adjusted_aspect_score"]
    * preference_mode["sampled_weight"]
)

In [11]:
def create_simulation_rankings(
    simulation_data,
    products,
    max_budget
):

    scores = (
        simulation_data
        .groupby(
            ["simulation_id", "product_id"],
            as_index=False
        )
        ["weighted_contribution"]
        .sum()
        .rename(
            columns={
                "weighted_contribution":
                "personalized_score"
            }
        )
    )

    scores = scores.merge(
        products,
        on="product_id",
        how="left"
    )

    scores = scores[
        scores["price"] <= max_budget
    ].copy()

    scores = scores.sort_values(
        [
            "simulation_id",
            "personalized_score"
        ],
        ascending=[True, False]
    )

    scores["rank"] = (
        scores
        .groupby("simulation_id")
        .cumcount() + 1
    )

    return scores

In [12]:
preference_rankings = create_simulation_rankings(
    preference_mode,
    products,
    max_budget
)

print(
    "Preference simulations:",
    preference_rankings["simulation_id"].nunique()
)

display(preference_rankings.head(8))

Preference simulations: 500


,simulation_id,product_id,personalized_score,product_name,category,price,rank
0,0,P001,0.822707,Phone A,smartphone,165000,1
4,0,P005,0.821986,Phone E,smartphone,160000,2
1,0,P002,0.811235,Phone B,smartphone,150000,3
3,0,P004,0.776217,Phone D,smartphone,140000,4
9,1,P005,0.825542,Phone E,smartphone,160000,1
5,1,P001,0.811548,Phone A,smartphone,165000,2
6,1,P002,0.807780,Phone B,smartphone,150000,3
8,1,P004,0.781897,Phone D,smartphone,140000,4


In [13]:
evidence_mode = features.merge(
    nominal_preferences[
        ["aspect", "nominal_weight"]
    ],
    on="aspect",
    how="inner"
)

evidence_mode = evidence_mode.merge(
    nominal_evidence,
    on=["product_id", "aspect"],
    how="inner"
)

evidence_mode = evidence_mode.merge(
    evidence_samples,
    on=["product_id", "aspect"],
    how="inner"
)

evidence_mode["evidence_factor"] = (
    evidence_mode["evidence_reliability"]
    / evidence_mode["nominal_evidence_reliability"]
)

evidence_mode["adjusted_aspect_score"] = np.clip(
    evidence_mode["nominal_aspect_score"]
    * evidence_mode["evidence_factor"],
    0,
    1
)

evidence_mode["weighted_contribution"] = (
    evidence_mode["adjusted_aspect_score"]
    * evidence_mode["nominal_weight"]
)

In [14]:
evidence_rankings = create_simulation_rankings(
    evidence_mode,
    products,
    max_budget
)

print(
    "Evidence simulations:",
    evidence_rankings["simulation_id"].nunique()
)

display(evidence_rankings.head(8))

Evidence simulations: 500


,simulation_id,product_id,personalized_score,product_name,category,price,rank
1,0,P002,0.847871,Phone B,smartphone,150000,1
4,0,P005,0.825845,Phone E,smartphone,160000,2
0,0,P001,0.799899,Phone A,smartphone,165000,3
3,0,P004,0.784776,Phone D,smartphone,140000,4
9,1,P005,0.817740,Phone E,smartphone,160000,1
5,1,P001,0.810151,Phone A,smartphone,165000,2
6,1,P002,0.807854,Phone B,smartphone,150000,3
8,1,P004,0.800303,Phone D,smartphone,140000,4


In [15]:
joint_mode = features.merge(
    preference_samples[
        [
            "simulation_id",
            "aspect",
            "sampled_weight"
        ]
    ],
    on="aspect",
    how="inner"
)

joint_mode = joint_mode.merge(
    nominal_evidence,
    on=["product_id", "aspect"],
    how="inner"
)

joint_mode = joint_mode.merge(
    evidence_samples,
    on=[
        "simulation_id",
        "product_id",
        "aspect"
    ],
    how="inner"
)

joint_mode["evidence_factor"] = (
    joint_mode["evidence_reliability"]
    / joint_mode["nominal_evidence_reliability"]
)

joint_mode["adjusted_aspect_score"] = np.clip(
    joint_mode["nominal_aspect_score"]
    * joint_mode["evidence_factor"],
    0,
    1
)

joint_mode["weighted_contribution"] = (
    joint_mode["adjusted_aspect_score"]
    * joint_mode["sampled_weight"]
)

In [16]:
joint_rankings = create_simulation_rankings(
    joint_mode,
    products,
    max_budget
)

print(
    "Joint simulations:",
    joint_rankings["simulation_id"].nunique()
)

display(joint_rankings.head(8))

Joint simulations: 500


,simulation_id,product_id,personalized_score,product_name,category,price,rank
1,0,P002,0.848780,Phone B,smartphone,150000,1
4,0,P005,0.823685,Phone E,smartphone,160000,2
0,0,P001,0.808376,Phone A,smartphone,165000,3
3,0,P004,0.788137,Phone D,smartphone,140000,4
9,1,P005,0.819333,Phone E,smartphone,160000,1
5,1,P001,0.810879,Phone A,smartphone,165000,2
6,1,P002,0.805581,Phone B,smartphone,150000,3
8,1,P004,0.803508,Phone D,smartphone,140000,4


In [17]:
def top1_stability(
    rankings,
    nominal_winner
):

    top1 = rankings[
        rankings["rank"] == 1
    ]

    retained = (
        top1["product_id"]
        == nominal_winner
    ).sum()

    stability = (
        retained
        / top1["simulation_id"].nunique()
    )

    return stability

In [18]:
preference_stability = top1_stability(
    preference_rankings,
    nominal_winner
)

evidence_stability = top1_stability(
    evidence_rankings,
    nominal_winner
)

joint_stability = top1_stability(
    joint_rankings,
    nominal_winner
)

stability_summary = pd.DataFrame({
    "mode": [
        "Preference-only",
        "Evidence-only",
        "Joint"
    ],
    "top1_stability": [
        preference_stability,
        evidence_stability,
        joint_stability
    ]
})

display(stability_summary)

,mode,top1_stability
0,Preference-only,0.914
1,Evidence-only,0.478
2,Joint,0.478


In [19]:
def winner_summary(rankings, mode):

    top1 = rankings[
        rankings["rank"] == 1
    ]

    summary = (
        top1
        .groupby(
            ["product_id", "product_name"]
        )
        .size()
        .reset_index(name="top1_count")
    )

    summary["top1_probability"] = (
        summary["top1_count"]
        / top1["simulation_id"].nunique()
    )

    summary["mode"] = mode

    return summary


winner_comparison = pd.concat(
    [
        winner_summary(
            preference_rankings,
            "Preference-only"
        ),
        winner_summary(
            evidence_rankings,
            "Evidence-only"
        ),
        winner_summary(
            joint_rankings,
            "Joint"
        )
    ],
    ignore_index=True
)

display(winner_comparison)

,product_id,product_name,top1_count,top1_probability,mode
0,P001,Phone A,6,0.012,Preference-only
1,P002,Phone B,37,0.074,Preference-only
2,P005,Phone E,457,0.914,Preference-only
3,P001,Phone A,129,0.258,Evidence-only
4,P002,Phone B,129,0.258,Evidence-only
5,P004,Phone D,3,0.006,Evidence-only
6,P005,Phone E,239,0.478,Evidence-only
7,P001,Phone A,129,0.258,Joint
8,P002,Phone B,128,0.256,Joint
9,P004,Phone D,4,0.008,Joint


In [20]:
assert (
    preference_rankings["simulation_id"].nunique()
    == 500
)

assert (
    evidence_rankings["simulation_id"].nunique()
    == 500
)

assert (
    joint_rankings["simulation_id"].nunique()
    == 500
)

assert (
    stability_summary["top1_stability"]
    .between(0, 1)
    .all()
)

print(
    "MONTE CARLO SIMULATION TESTS: PASSED"
)

MONTE CARLO SIMULATION TESTS: PASSED


In [21]:
OUTPUT_DIR.mkdir(exist_ok=True)

stability_summary.to_csv(
    OUTPUT_DIR /
    "monte_carlo_stability_summary.csv",
    index=False
)

winner_comparison.to_csv(
    OUTPUT_DIR /
    "monte_carlo_winner_summary.csv",
    index=False
)

print("Monte Carlo outputs saved.")

Monte Carlo outputs saved.


In [22]:
nominal_rank_map = dict(
    zip(
        nominal_scores["product_id"],
        nominal_scores["rank"]
    )
)

print(nominal_rank_map)

{'P005': 1, 'P001': 2, 'P002': 3, 'P004': 4}


In [23]:
from scipy.stats import kendalltau


def mean_kendall_tau(
    rankings,
    nominal_rank_map
):
    taus = []

    product_ids = list(
        nominal_rank_map.keys()
    )

    nominal_ranks = [
        nominal_rank_map[p]
        for p in product_ids
    ]

    for simulation_id, group in rankings.groupby(
        "simulation_id"
    ):

        simulated_map = dict(
            zip(
                group["product_id"],
                group["rank"]
            )
        )

        simulated_ranks = [
            simulated_map[p]
            for p in product_ids
        ]

        tau, _ = kendalltau(
            nominal_ranks,
            simulated_ranks
        )

        taus.append(tau)

    return np.mean(taus)

In [24]:
preference_tau = mean_kendall_tau(
    preference_rankings,
    nominal_rank_map
)

evidence_tau = mean_kendall_tau(
    evidence_rankings,
    nominal_rank_map
)

joint_tau = mean_kendall_tau(
    joint_rankings,
    nominal_rank_map
)

print(
    "Preference-only Mean Kendall Tau:",
    round(preference_tau, 4)
)

print(
    "Evidence-only Mean Kendall Tau:",
    round(evidence_tau, 4)
)

print(
    "Joint Mean Kendall Tau:",
    round(joint_tau, 4)
)

Preference-only Mean Kendall Tau: 0.8193
Evidence-only Mean Kendall Tau: 0.5073
Joint Mean Kendall Tau: 0.4987


In [25]:
def calculate_rank_entropy(rankings):

    n_products = (
        rankings["product_id"].nunique()
    )

    product_entropies = []

    for product_id, group in rankings.groupby(
        "product_id"
    ):

        probabilities = (
            group["rank"]
            .value_counts(normalize=True)
            .values
        )

        entropy = -np.sum(
            probabilities
            * np.log(probabilities)
        )

        if n_products > 1:
            normalized_entropy = (
                entropy / np.log(n_products)
            )
        else:
            normalized_entropy = 0.0

        product_entropies.append({
            "product_id": product_id,
            "rank_entropy":
                normalized_entropy
        })

    return pd.DataFrame(
        product_entropies
    )

In [26]:
preference_entropy = (
    calculate_rank_entropy(
        preference_rankings
    )
)

evidence_entropy = (
    calculate_rank_entropy(
        evidence_rankings
    )
)

joint_entropy = (
    calculate_rank_entropy(
        joint_rankings
    )
)

entropy_summary = pd.DataFrame({
    "mode": [
        "Preference-only",
        "Evidence-only",
        "Joint"
    ],
    "mean_rank_entropy": [
        preference_entropy[
            "rank_entropy"
        ].mean(),

        evidence_entropy[
            "rank_entropy"
        ].mean(),

        joint_entropy[
            "rank_entropy"
        ].mean()
    ]
})

display(entropy_summary)

,mode,mean_rank_entropy
0,Preference-only,0.348332
1,Evidence-only,0.788547
2,Joint,0.792971


In [27]:
full_stability_summary = pd.DataFrame({
    "mode": [
        "Preference-only",
        "Evidence-only",
        "Joint"
    ],

    "top1_stability": [
        preference_stability,
        evidence_stability,
        joint_stability
    ],

    "mean_kendall_tau": [
        preference_tau,
        evidence_tau,
        joint_tau
    ],

    "mean_rank_entropy": [
        preference_entropy[
            "rank_entropy"
        ].mean(),

        evidence_entropy[
            "rank_entropy"
        ].mean(),

        joint_entropy[
            "rank_entropy"
        ].mean()
    ]
})

display(full_stability_summary)

,mode,top1_stability,mean_kendall_tau,mean_rank_entropy
0,Preference-only,0.914,0.819333,0.348332
1,Evidence-only,0.478,0.507333,0.788547
2,Joint,0.478,0.498667,0.792971


In [28]:
evidence_top1 = (
    evidence_rankings[
        evidence_rankings["rank"] == 1
    ]
    .sort_values("simulation_id")
)

joint_top1 = (
    joint_rankings[
        joint_rankings["rank"] == 1
    ]
    .sort_values("simulation_id")
)

same_winner_rate = np.mean(
    evidence_top1["product_id"].values
    ==
    joint_top1["product_id"].values
)

print(
    "Evidence-only vs Joint same winner rate:",
    round(same_winner_rate, 4)
)

Evidence-only vs Joint same winner rate: 0.878


In [29]:
assert -1 <= preference_tau <= 1
assert -1 <= evidence_tau <= 1
assert -1 <= joint_tau <= 1

assert (
    full_stability_summary[
        "mean_rank_entropy"
    ]
    .between(0, 1)
    .all()
)

print(
    "RANKING STABILITY TESTS: PASSED"
)

RANKING STABILITY TESTS: PASSED


In [30]:
def instability_score(
    top1_stability,
    mean_kendall_tau,
    mean_rank_entropy
):
    top1_instability = 1 - top1_stability

    # Kendall Tau ranges from -1 to +1.
    # Convert it to a 0-1 instability measure.
    kendall_instability = (
        1 - mean_kendall_tau
    ) / 2

    entropy_instability = mean_rank_entropy

    return np.mean([
        top1_instability,
        kendall_instability,
        entropy_instability
    ])

In [31]:
preference_instability = instability_score(
    preference_stability,
    preference_tau,
    preference_entropy["rank_entropy"].mean()
)

evidence_instability = instability_score(
    evidence_stability,
    evidence_tau,
    evidence_entropy["rank_entropy"].mean()
)

joint_instability = instability_score(
    joint_stability,
    joint_tau,
    joint_entropy["rank_entropy"].mean()
)

print(
    "Preference instability:",
    round(preference_instability, 4)
)

print(
    "Evidence instability:",
    round(evidence_instability, 4)
)

print(
    "Joint instability:",
    round(joint_instability, 4)
)

Preference instability: 0.1749
Evidence instability: 0.519
Joint instability: 0.5219


In [32]:
def classify_uncertainty_source(
    preference_instability,
    evidence_instability,
    stable_threshold=0.20,
    source_margin=0.10
):
    # Both sources have low impact
    if (
        preference_instability <= stable_threshold
        and evidence_instability <= stable_threshold
    ):
        return "Stable", "Recommend"

    difference = (
        preference_instability
        - evidence_instability
    )

    # Preference uncertainty clearly dominates
    if difference > source_margin:
        return (
            "Preference-sensitive",
            "Clarify"
        )

    # Evidence uncertainty clearly dominates
    if difference < -source_margin:
        return (
            "Evidence-sensitive",
            "Warn"
        )

    # Both have similar meaningful impact
    return (
        "Mixed",
        "Combine"
    )

In [33]:
state, action = classify_uncertainty_source(
    preference_instability,
    evidence_instability
)

print("Detected state:", state)
print("Recommended action:", action)

Detected state: Evidence-sensitive
Recommended action: Warn
